In [6]:
# For sweet Crumbs business
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.create("Sweet Crumbs Orders")      # run once; next time use gc.open("Sweet Crumbs Orders")
ws = sh.sheet1
ws.append_row(["name", "item", "qty", "date", "eggless", "total"])
print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1m62GBz1-G4vFgxpnqfgeaT_a4sf6mb9laL1vmhxTsFM


In [3]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}
HOURS = {"monday": "9 am to 8 pm", "tuesday": "9 am to 8 pm", "wednesday": "9 am to 8 pm",
         "thursday": "9 am to 8 pm", "friday": "9 am to 8 pm", "saturday": "9 am to 8 pm",
         "sunday": "10 am to 6 pm"}

class Order(BaseModel):
    """A cake order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["plum cake", "chocolate cake", "cupcake"] = Field(description="Menu item")
    qty: int = Field(description="How many", ge=1)
    date: str = Field(description="Pickup date as the customer said it")
    eggless: bool = Field(description="True if the customer asked for eggless")

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one menu item.

    Args:
        item: Menu item, e.g. "plum cake", "chocolate cake" or "cupcake"
        qty: How many the customer wants
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed cake order to the Orders sheet.
    Use only when the customer has given their name, item, quantity and pickup date."""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, date=date, eggless=eggless)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.date,
                   "yes" if order.eggless else "no", total])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."

In [4]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly order assistant for Sweet Crumbs Home Bakery.
FACTS:  Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Always use get_price for prices and is_open for timings; never guess them.
        Before log_order, make sure you have name, item, quantity and pickup date; ask if missing.
        After saving, read the order back with the total.
FORMAT: Under 60 words, friendly, English or Telugu matching the customer."""

agent = create_agent(model=llm, tools=[get_price, is_open, log_order], system_prompt=SYSTEM)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print(chat("Are you open on Sunday? How much are 2 chocolate cakes?"))
print(chat("I'm Ravi. Please book 2 eggless chocolate cakes for 24 Dec."))

Yes, we’re open Sunday from 10 am to 6 pm. Two chocolate cakes cost Rs 1300 in total. Let me know your name, quantity, and pickup date to place the order!
✅ Order confirmed!  
2 eggless chocolate cakes for Ravi, pickup 24 Dec. Total: Rs 1300.  
See you then—happy baking!


In [5]:
for m in history:
    m.pretty_print()

================================ Human Message =================================

Are you open on Sunday? How much are 2 chocolate cakes?
================================== Ai Message ==================================
Tool Calls:
  is_open (fc_eb8b6540-31f2-49e2-9a8a-03d5e8146b39)
 Call ID: fc_eb8b6540-31f2-49e2-9a8a-03d5e8146b39
  Args:
    day: sunday
================================= Tool Message =================================
Name: is_open

Open 10 am to 6 pm on sunday.
================================== Ai Message ==================================
Tool Calls:
  get_price (fc_213f45c5-4b63-4bc6-b599-4845bf6bccf1)
 Call ID: fc_213f45c5-4b63-4bc6-b599-4845bf6bccf1
  Args:
    item: chocolate cake
    qty: 2
================================= Tool Message =================================
Name: get_price

2 x chocolate cake = Rs 1300
================================== Ai Message ==================================

Yes, we’re open Sunday from 10 am to 6 pm. Two chocolate cakes cost

In [ ]:
# Turning it into my business requirement "PrintSmart Printing Shop"

In [52]:
# For printsmart Printing shop business
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.create("Print Orders")
ws = sh.sheet1
ws.append_row(["name", "no. of pages", "print type", "paper size","both_side", "pickup date", "total"])
print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1CL0P9mbtM1MjOtsBdY-h8kQpSGgwstAgyjog6HMHw8w


In [55]:
from langchain.tools import tool

MENU = {
    "black and white": {
        "single": 1,
        "double": 2
    },
    "color": 10
}

@tool
def get_printing_price(
    pages: int,
    print_type: str,
    both_side: bool = False
) -> str:
    """
    Calculate the total printing price.

    Args:
        pages: Number of pages.
        print_type: black and white or color.
        both_side: True for double-sided, False for single-sided.
    """

    print_type = print_type.lower().strip()

    if pages <= 0:
        return "Pages must be greater than zero."

    if print_type == "black and white":
        price = MENU["black and white"]["double" if both_side else "single"]

    elif print_type == "color":
        price = MENU["color"]

    else:
        return "Unsupported printing type."

    total = pages * price

    return f"Total printing cost is Rs. {total}"


@tool
def get_printing_options() -> str:
    """Get the printing services and document formats supported
    by PrintSmart Printing Shop.

    Use this when a customer asks about available printing
    services, paper sizes or accepted file formats.
    """
    return """
    Printing services:
    - Black and White single-sided
    - Black and White double-sided
    - Color printing

    Supported paper sizes: A4, A3, A5
    Accepted document format: PDF
    Collection: Shop pickup only
    Home delivery: Not available
    """

from langchain.tools import tool

@tool
def log_print_order(
    name: str,
    no_of_pages: int,
    print_type: str,
    paper_size: str,
    both_side: bool,
    pickup_date: str
) -> str:
    """
    Save a confirmed printing order to Google Sheets.
    Calculate the total price and record all order details.
    Call only after customer confirmation.

    Args:
        name: Customer name.
        no_of_pages: Number of pages.
        print_type: black and white or color.
        paper_size: A4, A3 or A5.
        both_side: True for double-sided, False for single-sided.
        pickup_date: Customer's pickup date.
    """

    try:
        order = PrintOrder(
            name=name,
            no_of_pages=no_of_pages,
            print_type=print_type.lower().strip(),
            paper_size=paper_size.upper().strip(),
            both_side=both_side,
            pickup_date=pickup_date
        )

        if order.print_type == "black and white":
            price = MENU["black and white"][
                "double" if order.both_side else "single"
            ]
        else:
            price = MENU["color"]

        total = order.no_of_pages * price

        ws.append_row([
            order.name,
            order.no_of_pages,
            order.print_type,
            order.paper_size,
            order.both_side,
            str(order.pickup_date),
            total
        ])

        return f"Order saved successfully. Total: Rs. {total}"

    except Exception as e:
        return f"Could not save order: {str(e)}"

In [56]:
from pydantic import BaseModel, Field
from typing import Literal
from datetime import date

class PrintOrder(BaseModel):
    name: str = Field(description="Customer name")

    no_of_pages: int = Field(
        description="Number of pages",
        ge=1
    )

    print_type: Literal[
        "black and white",
        "color"
    ] = Field(description="Printing type")

    paper_size: Literal[
        "A4", "A3", "A5"
    ]

    both_side: bool = Field(
        description="True for double-sided, False for single-sided"
    )

    pickup_date: date

In [57]:
extractor = llm.with_structured_output(PrintOrder)

In [65]:

from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are the friendly customer support assistant
for PrintSmart Printing Shop.

FACTS:
- We accept PDF files only.
- Supported sizes: A4, A3, A5.
- B&W single-sided: Rs. 1 per page.
- B&W double-sided: Rs. 2 per page.
- Color printing: Rs. 10 per page.
- Shop pickup only.
- Home delivery is not available.

RULES:
-For black and white printing, identify whether the customer wants single-sided or double-sided printing.
- Set both_side to True for double-sided printing.
- Set both_side to False for single-sided printing.
- For color printing, both_side is not applicable,Always set both_side to False.
- Use get_printing_price for price calculations.
- Ask for clarification when information is missing.
- Confirm the order and log it using log_print_order once all details are collected

FORMAT:
Be friendly, concise and professional and answer under 50 words.
"""

agent = create_agent(
    model=llm,
    tools=[
        get_printing_price,
        get_printing_options,
        log_print_order
    ],
    system_prompt=SYSTEM
)


In [67]:
print(chat("Hi iam ram, book me black and white printing for 50 pages double side on A4 sheets on 28 NOV 2026?"))

✅ Order confirmed!  
- **Name:** Ram  
- **Pages:** 50 double‑sided B&W  
- **Paper size:** A4  
- **Pickup date:** 28 Nov 2026  
- **Total:** ₹100  

See you at the shop on the pickup date. Let me know if anything changes!


In [68]:
print(chat("Do you support A3 size paper"))

Yes, we print on A3 paper (as well as A4 and A5). We accept PDFs and offer B&W single‑/double‑sided or color printing. Pick‑up only—no home delivery. Let me know what you’d like to print!


In [69]:
print(chat("I want to know the total price of 100 pages as black and white single side and 25 colour prints"))

The total comes to **Rs 350**:

- 100 B&W single‑sided pages × Rs 1 = Rs 100  
- 25 color pages × Rs 10 = Rs 250  

Let me know if you’d like to place the order!


In [70]:
history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

In [72]:
print(chat("Hi iam Varun, book me black and white printing for 200 pages double side on A4 sheets on 30 NOV 2026"))

Sure thing, Varun! I’ve calculated the cost: Rs. 400 for 200 double‑sided B&W pages on A4.  
Could you please confirm the pickup date in YYYY‑MM‑DD format (e.g., 2026-11-30)? Once I have that, I’ll log your order.


In [75]:
print(chat("Confirm on 2026-11-30"))

All set! Your order is confirmed for 200 double‑sided B&W pages on A4, pickup on 30 Nov 2026. Total cost: Rs. 400. Thank you for choosing PrintSmart—see you soon!


In [77]:
for m in history:
    m.pretty_print()

================================ Human Message =================================

Hi iam Varun, book me black and white printing for 200 pages double side on A4 sheets on 30 NOV 2026
================================== Ai Message ==================================
Tool Calls:
  get_printing_price (fc_08386c68-8e61-4adf-b5e6-6617a1029c54)
 Call ID: fc_08386c68-8e61-4adf-b5e6-6617a1029c54
  Args:
    both_side: True
    pages: 200
    print_type: black and white
================================= Tool Message =================================
Name: get_printing_price

Total printing cost is Rs. 400
================================== Ai Message ==================================
Tool Calls:
  log_print_order (fc_c672b6ca-7199-4c8b-ad01-98e1a9c2e5b0)
 Call ID: fc_c672b6ca-7199-4c8b-ad01-98e1a9c2e5b0
  Args:
    both_side: True
    name: Varun
    no_of_pages: 200
    paper_size: A4
    pickup_date: 30 NOV 2026
    print_type: black and white
================================= Tool Message ===